# Clarkson II preprocessing: first three steps

This notebook covers only:

1. Preserve the raw dataset unchanged.
2. Validate every raw file and record.
3. Generate normalized timestamp files without modifying the raw files.

The normalized output keeps the original 100-nanosecond timestamp, adds an absolute timestamp in seconds, and adds elapsed seconds from the first valid event in each user file.

In [1]:
from pathlib import Path
from collections import Counter
import csv
import gzip
import hashlib
import json
from datetime import datetime, timezone

# Locate the project root whether the notebook is launched from the project
# directory or from this preprocessing folder.
HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents]
                     if (p / 'clarkson2_preprocessing').is_dir()), HERE)

# The rename was blocked earlier, so support both names.
DATASET_CANDIDATES = [PROJECT_ROOT / 'clarkson2',
                      PROJECT_ROOT / 'filtered_dataset']
RAW_DIR = next((p for p in DATASET_CANDIDATES if p.is_dir()), None)
if RAW_DIR is None:
    raise FileNotFoundError('Could not find clarkson2 or filtered_dataset')

WORK_DIR = PROJECT_ROOT / 'clarkson2_preprocessing'
OUTPUT_DIR = WORK_DIR / 'outputs'
NORMALIZED_DIR = OUTPUT_DIR / 'normalized_events'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NORMALIZED_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Raw dataset :', RAW_DIR)
print('Output dir  :', OUTPUT_DIR)

Project root: C:\Users\keyar\Documents\GradSem3\Biometric\Term Project
Raw dataset : C:\Users\keyar\Documents\GradSem3\Biometric\Term Project\filtered_dataset
Output dir  : C:\Users\keyar\Documents\GradSem3\Biometric\Term Project\clarkson2_preprocessing\outputs


## Step 1 — preserve the raw data

The notebook reads from the original dataset directory and writes only to `clarkson2_preprocessing/outputs`. It never overwrites or renames raw files. The hash below records the source files used for the run.

In [2]:
raw_files = sorted(p for p in RAW_DIR.iterdir() if p.is_file())
if not raw_files:
    raise ValueError(f'No user files found in {RAW_DIR}')

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

source_manifest = {
    'raw_directory': str(RAW_DIR),
    'file_count': len(raw_files),
    'files': {p.name: {'bytes': p.stat().st_size, 'sha256': sha256_file(p)}
              for p in raw_files},
}
(OUTPUT_DIR / 'raw_manifest.json').write_text(
    json.dumps(source_manifest, indent=2), encoding='utf-8')

print(f'Raw files recorded: {len(raw_files)}')
print('Raw files remain untouched.')

Raw files recorded: 100
Raw files remain untouched.


## Step 2 — validate files and records

Expected row schema:

```text
timestamp_100ns<TAB>press_release_flag<TAB>key
```

This pass does not silently repair data. It counts malformed rows, invalid values, timestamp backsteps, and consecutive duplicate rows so that cleaning decisions remain auditable.

In [3]:
def validate_file(path):
    stats = Counter()
    stats['file'] = path.name
    stats['bytes'] = path.stat().st_size
    previous_timestamp = None
    previous_raw_line = None
    first_timestamp = None
    last_timestamp = None

    with path.open('r', encoding='utf-8', errors='replace', newline='') as handle:
        for line_number, raw_line in enumerate(handle, start=1):
            stats['rows'] += 1
            line = raw_line.rstrip('\r\n')
            if line == previous_raw_line:
                stats['consecutive_duplicate_rows'] += 1
            previous_raw_line = line

            fields = line.split('\t')
            if len(fields) != 3:
                stats['malformed_columns'] += 1
                continue

            timestamp_text, flag_text, key = fields
            if not key:
                stats['empty_keys'] += 1
                continue
            try:
                timestamp = int(timestamp_text)
            except ValueError:
                stats['invalid_timestamps'] += 1
                continue
            if flag_text not in {'0', '1'}:
                stats['invalid_flags'] += 1
                continue
            if timestamp < 0:
                stats['negative_timestamps'] += 1
            if previous_timestamp is not None:
                if timestamp < previous_timestamp:
                    stats['timestamp_backsteps'] += 1
                elif timestamp == previous_timestamp:
                    stats['equal_timestamps'] += 1
            previous_timestamp = timestamp
            first_timestamp = timestamp if first_timestamp is None else first_timestamp
            last_timestamp = timestamp
            stats['valid_rows'] += 1
            stats[f'flag_{flag_text}'] += 1

    stats['first_timestamp_100ns'] = first_timestamp
    stats['last_timestamp_100ns'] = last_timestamp
    if first_timestamp is not None and last_timestamp is not None:
        stats['duration_seconds_from_endpoints'] = (last_timestamp - first_timestamp) / 10_000_000
    return dict(stats)

validation_rows = [validate_file(path) for path in raw_files]
with (OUTPUT_DIR / 'validation_report.json').open('w', encoding='utf-8') as handle:
    json.dump(validation_rows, handle, indent=2)

field_names = ['file', 'rows', 'valid_rows', 'malformed_columns', 'invalid_timestamps',
               'invalid_flags', 'empty_keys', 'negative_timestamps',
               'timestamp_backsteps', 'equal_timestamps',
               'consecutive_duplicate_rows', 'flag_0', 'flag_1',
               'duration_seconds_from_endpoints']
with (OUTPUT_DIR / 'validation_report.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=field_names, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(validation_rows)

totals = Counter()
for row in validation_rows:
    for key, value in row.items():
        if key not in {'file', 'bytes', 'first_timestamp_100ns', 'last_timestamp_100ns', 'duration_seconds_from_endpoints'}:
            totals[key] += value if isinstance(value, int) else 0

print('Files:', len(validation_rows))
print('Total rows:', totals['rows'])
print('Valid rows:', totals['valid_rows'])
print('Malformed rows:', totals['malformed_columns'])
print('Invalid timestamps:', totals['invalid_timestamps'])
print('Invalid flags:', totals['invalid_flags'])
print('Timestamp backsteps:', totals['timestamp_backsteps'])
print('Consecutive duplicate rows:', totals['consecutive_duplicate_rows'])

Files: 100
Total rows: 25261528
Valid rows: 25261511
Malformed rows: 17
Invalid timestamps: 0
Invalid flags: 0
Timestamp backsteps: 21
Consecutive duplicate rows: 395335


## Step 3 — normalize timestamps

For every valid row, the output keeps the original integer timestamp and adds:

- `timestamp_seconds`: absolute timestamp divided by 10,000,000.
- `elapsed_seconds`: time from the first valid event in that user file.

Invalid rows are skipped in the normalized copy and remain documented in the validation report. The output is compressed TSV so the raw files are not duplicated in an uncompressed form.

In [4]:
def normalize_file(source_path, destination_path):
    valid_rows = 0
    skipped_rows = 0
    first_timestamp = None

    with source_path.open('r', encoding='utf-8', errors='replace', newline='') as source, \
         gzip.open(destination_path, 'wt', encoding='utf-8', newline='') as destination:
        writer = csv.writer(destination, delimiter='\t', lineterminator='\n')
        writer.writerow(['timestamp_100ns', 'timestamp_seconds', 'elapsed_seconds', 'flag', 'key'])
        for raw_line in source:
            fields = raw_line.rstrip('\r\n').split('\t')
            if len(fields) != 3:
                skipped_rows += 1
                continue
            timestamp_text, flag_text, key = fields
            try:
                timestamp = int(timestamp_text)
            except ValueError:
                skipped_rows += 1
                continue
            if timestamp < 0 or flag_text not in {'0', '1'} or not key:
                skipped_rows += 1
                continue
            if first_timestamp is None:
                first_timestamp = timestamp
            absolute_seconds = timestamp / 10_000_000
            elapsed_seconds = (timestamp - first_timestamp) / 10_000_000
            writer.writerow([timestamp, f'{absolute_seconds:.7f}',
                             f'{elapsed_seconds:.7f}', flag_text, key])
            valid_rows += 1
    return valid_rows, skipped_rows

normalization_summary = []
for source_path in raw_files:
    destination_path = NORMALIZED_DIR / f'{source_path.name}.tsv.gz'
    valid_rows, skipped_rows = normalize_file(source_path, destination_path)
    normalization_summary.append({
        'file': source_path.name,
        'output': str(destination_path),
        'valid_rows_written': valid_rows,
        'skipped_rows': skipped_rows,
    })

(OUTPUT_DIR / 'normalization_summary.json').write_text(
    json.dumps(normalization_summary, indent=2), encoding='utf-8')

print(f'Normalized files written: {len(normalization_summary)}')
print('Output directory:', NORMALIZED_DIR)

Normalized files written: 100
Output directory: C:\Users\keyar\Documents\GradSem3\Biometric\Term Project\clarkson2_preprocessing\outputs\normalized_events


## Output files

The notebook creates:

- `outputs/raw_manifest.json` — raw file sizes and SHA-256 hashes
- `outputs/validation_report.csv` — one validation summary per user
- `outputs/validation_report.json` — detailed validation report
- `outputs/normalization_summary.json` — rows written and skipped
- `outputs/normalized_events/*.tsv.gz` — normalized event streams

No hold-time pairing, burst segmentation, outlier removal, feature scaling, or train/test splitting is performed yet. Those belong to the next preprocessing stage.